In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
!uv pip install lightgbm scikit-learn pandas pyarrow matplotlib --quiet

In [ ]:
import lightgbm as lgbm
from pomap.interface import leaf, feed, ensemble

import polars as pl
import numpy as np
import datetime as dt

from functools import partial

In [ ]:
t = (
    pl.datetime_range(
        start=dt.datetime(2025, 1, 1),
        end=dt.datetime(2026, 1, 1),
        interval="15m",
        eager=True,
    )
    .rename("t")
    .to_frame()
)
x = np.random.normal(size=t.shape[0]) * np.pi
true_y = np.sin(x)
z = np.random.normal(scale=0.2, size=t.shape[0])
observed_y = true_y + z

data = t.with_columns(x=x, observed_y=observed_y, true_y=true_y)

In [ ]:
class LGBMWrapper:
    def __init__(self, features, target, **kwargs):
        self.model = lgbm.LGBMRegressor(**kwargs)
        self.features = features
        self.target = target

    def fit(self, training_set: pl.DataFrame, validation_set: pl.DataFrame = None):

        X_train = training_set.select(*self.features).to_pandas()
        y_train = training_set[self.target].to_pandas()

        eval_set = [(X_train, y_train)]
        eval_names = ["training"]

        if validation_set is not None:
            X_val = validation_set.select(*self.features).to_pandas()
            y_val = validation_set[self.target].to_pandas()
            eval_set += [(X_val, y_val)]
            eval_names += ["validation"]

        self.model.fit(X_train, y_train, eval_set=eval_set, eval_names=eval_names)

    def predict(self, df: pl.DataFrame):
        X = df.select(*self.features).to_pandas()
        return self.model.predict(X)

In [ ]:
teacher_model = partial(
    LGBMWrapper,
    features=["x"],
    target="observed_y",
    num_leaves=31,
    max_depth=5,
    learning_rate=0.01,
    n_estimators=10000,
)

student_model = partial(
    LGBMWrapper,
    features=["x"],
    target="observed_y",
    num_leaves=31,
    max_depth=5,
    learning_rate=0.1,
    n_estimators=10,
)

In [ ]:
teacher = leaf(teacher_model, "teacher")
student = leaf(student_model, "student")
untutored_student = leaf(student_model, "untutored_student")

In [ ]:
distillation = feed("distillation", source=teacher, consumer=student)
full_experiment = ensemble("full", distillation, untutored_student)

In [ ]:
full_experiment.fit(data);

In [ ]:
fitted = full_experiment.predict(data)

In [ ]:
ax = fitted.sort("x").to_pandas().plot(x="x", y="teacher")
fitted.sort("x").to_pandas().plot(
    x="x", y="teacher", kind="scatter", c="pink", alpha=0.3, ax=ax
)
fitted.sort("x").to_pandas().plot(
    x="x", y="student", kind="scatter", c="green", alpha=0.3, ax=ax
)
fitted.sort("x").to_pandas().plot(
    x="x", y="untutored_student", kind="scatter", c="blue", alpha=0.3, ax=ax
)